# Draw IO sandbox
Requires N2G and ttp installed in /lib folder.
These libraries are not available from official Anaconda or pip distribution channels.

Check out `igraph <https://igraph.org/python/doc/tutorial/tutorial.html#layout-algorithms>`

In [ ]:
source = '/Users/bue/dev/borr/2021-07-14 BORR IM/DB/DC-IM.json'

In [ ]:
import os
import sys
import json
from lxml import etree

In [ ]:
sys.path.insert(0, os.path.abspath('../../lib/N2G'))
sys.path.insert(1, os.path.abspath('../../lib/ttp'))

from N2G import drawio_diagram as dio
assert dio(), f"Unable to create draw io diagrams"

In [ ]:
with open(source, 'r') as src:
    ssot = json.load(src)
assert len(list(ssot['diagrams'])) > 0, f"Cannot find any diagram in {ssot['model']}"

In [ ]:
diagram = ssot['diagrams']['DIAG1807'] # Product
lang = ssot['model']['language']

In [ ]:
def entity_name(key: str) -> str:
    global ssot, lang
    enti = ssot['entities'][key]
    sn = enti['name'][lang]
    return sn

In [ ]:
def entity_description(key: str) -> str:
    global ssot, lang
    enti = ssot['entities'][key]
    descr = enti['descr'][lang]
    if isinstance(descr, str):
        return descr
    else:
        #print(f"Entity {key} has no description!\n{enti}")
        return None

In [ ]:

def entity_synonyms(key: str) -> str:
    global ssot, lang
    enti = ssot['entities'][key]
    syno = list(enti['synonyms'].values())
    if len(syno) > 0:
        synonym = syno[0].get(lang)
        if isinstance(synonym, str):
            return synonym
    return None

In [ ]:
entity_name('ENTI387')

In [ ]:
diagram_xml = f"""<?xml version="1.0" encoding="UTF-8"?>
<mxfile host="Electron" modified="2021-07-20T12:02:15.557Z" agent="curl/7.1" etag="25mQkM6mx7LJW4tu3GDx" version="14.6.13" type="device">
  <diagram id="-IuDeWdp_pBGzQphX35I" name="{diagram['name']}">
    <mxGraphModel dx="{diagram['width']}" dy="{diagram['height']}" pageWidth="{diagram['width']}" pageHeight="{diagram['height']}" grid="1" gridSize="10" guides="1" tooltips="1" connect="1" arrows="1" fold="1" page="1" pageScale="1" math="0" shadow="0">
      <root>
        <mxCell id="0" />
        <mxCell id="1" parent="0" />
      </root>
    </mxGraphModel>
  </diagram>
</mxfile>"""

In [ ]:
from io import BytesIO

parser = etree.XMLParser(remove_blank_text=True)
dom = etree.parse(BytesIO(diagram_xml.encode('utf-8')), parser)

#dom = etree.fromstring(diagram_xml.encode('utf-8'), remove_blank_text=True)

In [ ]:
root = dom.find('.//root')

In [ ]:
len(root.getchildren())

In [ ]:
import spectra
black = spectra.html('#000000')

In [ ]:
entity_xml = """<mxcell id="{id}" value="{name}" sytle="{style}" parent="1" vertex="1">"""
entity_style = "rounded=1;whiteSpace=wrap;html=1;align=center;verticalAlign=top;"

In [ ]:
visible = set( element['element'] for element in diagram['elements']['entity'])
len(visible), list(visible)[:2]

In [ ]:
for element in diagram['elements']['entity']:
    enti_key = element['element']
    enti = ssot['entities'][enti_key]
    
    uo = etree.Element('UserObject')
    uo.set('id', enti_key)
    uo.set('label', entity_name(enti_key))
    uo.set('link', 'ssot:' + enti_key)
    descr = entity_description(enti_key)
    if descr:
        uo.set('Beschreibung', descr)
    
    synonyms = entity_synonyms(enti_key)
    if synonyms:
        uo.set('Synonyme', synonyms)
    
    fillcolor = element['ui']['color']
    fill = spectra.html('#' + fillcolor)
    
    supertypes = enti.get('supertypes+')
    if len(supertypes) > 0 and len(visible.intersection(supertypes)) > 0:
        #print(f"Brightening up {enti_key}")
        fill = fill.brighten(amount=5)
    
    cell = etree.Element("mxCell", id=enti_key + '-cell', style=entity_style + f"fillColor={fill.hexcode};", parent='1', vertex='1')
    box = etree.Element("mxGeometry", x=str(element['pos_x']), y=str(element['pos_y']), 
                        width=str(element['ui']['width']), height=str(element['ui']['height']))
    box.set('as', 'geometry')
    
    cell.append(box)
    uo.append(cell)
    root.append(uo)

In [ ]:
graph = dom.find('.//mxGraphModel')

In [ ]:
import IPython
IPython.display.Code(etree.tostring(graph, pretty_print=True).decode('UTF-8'))